# 화음왕 — GPU 처리 시간 재기

mp3 한 곡을 멜로디 MIDI로 바꾸는 데 GPU에서 몇 초가 걸리는지 잽니다.

**먼저**: 위 메뉴 `런타임 → 런타임 유형 변경`에서 하드웨어 가속기를 **T4 GPU**로 고르세요. 그다음 셀을 위에서부터 차례로 실행합니다(셀 왼쪽의 ▶).

## 1. GPU 확인
`Tesla T4` 같은 이름이 나오면 됩니다. `GPU 없음`이 나오면 런타임 유형을 다시 확인하세요.

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음 — 런타임 유형을 T4 GPU로 바꾸세요')

## 2. 코드와 필요한 패키지 받기 (1~2분)

In [ ]:
!rm -rf qwer && git clone -q -b ccr-bfdf6d41-ijaztk --depth 1 https://github.com/hojung1231/qwer
!pip -q install torchcrepe demucs mido soundfile librosa pyworld
%cd qwer/tools/vocal2midi
print('준비 완료')

## 3. mp3 올리기
실행하면 `파일 선택` 버튼이 나옵니다. 시험할 곡 하나를 고르세요.

In [ ]:
from google.colab import files
up = files.upload()
SONG = list(up)[0]
print('올린 파일:', SONG)

## 4. 변환 실행
첫 실행은 모델을 내려받느라 조금 더 걸립니다. 정확한 시간을 보려면 **이 셀을 한 번 더 실행**한 뒤의 숫자를 보세요.

In [ ]:
import json, shutil
shutil.rmtree('out', ignore_errors=True)
!python3 vocal2midi.py "$SONG" -o out --separator demucs 2>&1 | grep -v -E 'mp3float|aist'
r = json.load(open('out/report.json'))
t = r['timings']
total = t.get('separation_s', 0) + t.get('pitch_tracking_s', 0) + t.get('notes_and_midi_s', 0)
print()
print('장치          :', t['device'])
print('곡 길이       : %.0f초' % t['audio_length_s'])
print('보컬 분리     : %.1f초' % t.get('separation_s', 0))
print('음높이 추적   : %.1f초' % t.get('pitch_tracking_s', 0))
print('음표화·MIDI   : %.1f초' % t.get('notes_and_midi_s', 0))
print('합계          : %.1f초' % total)
print('추정 조성/템포:', r['key'], '/', r['bpm'], 'BPM')

## 5. (선택) 화음 만들기 시간도 재기
가수 음색으로 화음 목소리를 합성하는 단계입니다. 이 단계는 GPU를 쓰지 않습니다.

In [ ]:
import time
t0 = time.time()
!python3 harmony.py out "$SONG" --pitch-source score --only A 2>&1 | grep -v -E 'mp3float|aist|pkg_resources' | head -3
print('화음 A 한 벌: %.1f초' % (time.time() - t0))

## 6. 결과 받기
MIDI와 분리한 육성을 내려받습니다.

In [ ]:
files.download('out/melody_quantised.mid')
files.download('out/vocal.wav')